In [41]:
# ...importing the necessary libraries and reading in the data.

import pandas as pd
import numpy as np

sales = pd.read_csv("C:/Users/gcart/OneDrive/Documents/dev/r/projs/book_reviews/solve_any_data_analysis_problem_projs/CH03_asboth_Solve_Any_Data_Analysis_Problem/data/purchases.csv")
print(sales.shape)

(71519, 11)


In [42]:
#  ...some missing guest information.

sales.isnull().sum()

event_time              0
product_id              0
category_id             0
category_code       16739
brand                5707
price                   0
session_id              0
customer_id         18448
guest_first_name    53071
guest_surname       53071
guest_postcode      53071
dtype: int64

In [43]:
# ...create a new column to track guest checkouts, which happen when a customer ID is not provided.

sales["is_guest"] = sales["customer_id"].isnull()

In [44]:
# The first line of this code checks for cases where the transaction was a guest checkout but we 
#     also had a customer ID,...
sales[sales["is_guest"] & sales["customer_id"].notnull()]

# ...and the second returns cases where we had neither.
sales[(sales["is_guest"] == False) & sales["customer_id"].isnull()]

,event_time,product_id,category_id,category_code,brand,price,session_id,customer_id,guest_first_name,guest_surname,guest_postcode,is_guest


In [45]:
# Counting the number of guest accounts is useful to determine what percentage of our customer data 
# will be “assumed” cases

sales["is_guest"].value_counts(normalize=True)

False    0.742055
True     0.257945
Name: is_guest, dtype: float64

In [46]:
# ... one row represents a purchased item, not a customer record, so the proportion of customers who 
# checked out as guests is not necessarily, calculate this actual proportion

guest_columns = ["guest_first_name", "guest_surname",
                 "guest_postcode"]
unique_guests = sales[guest_columns].drop_duplicates()
print(len(unique_guests))
unique_customers = sales["customer_id"].unique()
cust_total = len(unique_customers) + len(unique_guests)
print(len(unique_guests) / (cust_total-1))

8301
0.2495640671036017


In [47]:
# To export guests only we can filter using our is_guest column and export only the relevant columns.

guest_columns = ["guest_first_name", "guest_surname","guest_postcode", "is_guest"]
guests = sales.loc[sales["is_guest"], guest_columns]
guests = guests.drop_duplicates()
guests.head()

,guest_first_name,guest_surname,guest_postcode,is_guest
0,MICHAEL,MASON,RG497ZQ,True
2,COLE,WILKINSON,SW75TQ,True
3,MOHAMMED,RICHARDS,RG150RE,True
7,KIAN,MILLS,SW332TF,True
13,RUBY,OWEN,PO377YS,True


In [48]:
# For non-guest checkouts, we won’t have these columns, we will have only a customer ID:

non_guests = (
  pd.DataFrame(
    sales.loc[sales["customer_id"].notnull(), "customer_id"]
      .unique()
      .astype(int),
    columns=["customer_id"]
  )
)
non_guests.head()

,customer_id
0,7466
1,31266
2,534142828
3,1035
4,6985


In [49]:
# when we combine them we will have all the columns from both datasets and missing data where a 
# column did not exist in one of the datasets.

sales_customers = pd.concat([non_guests,guests], axis=0, ignore_index=True)
sales_customers.head()

,customer_id,guest_first_name,guest_surname,guest_postcode,is_guest
0,7466.0,NaN,NaN,NaN,NaN
1,31266.0,NaN,NaN,NaN,NaN
2,534142828.0,NaN,NaN,NaN,NaN
3,1035.0,NaN,NaN,NaN,NaN
4,6985.0,NaN,NaN,NaN,NaN


In [50]:
# First, we concatenate (or “union” if you are used to SQL terminology) our two datasets.
# Then we rename our columns and remove the guest prefix.

new_col_names = ["customer_id", "first_name", "surname", "postcode", "is_guest"]
sales_customers = sales_customers.set_axis(new_col_names, axis=1)
sales_customers.head()

,customer_id,first_name,surname,postcode,is_guest
0,7466.0,NaN,NaN,NaN,NaN
1,31266.0,NaN,NaN,NaN,NaN
2,534142828.0,NaN,NaN,NaN,NaN
3,1035.0,NaN,NaN,NaN,NaN
4,6985.0,NaN,NaN,NaN,NaN


In [51]:
# We also want to make sure we don’t have missing data, so we fill in the missing values 
#     for the is_guest column:

sales_customers["is_guest"] = sales_customers["is_guest"].fillna(False)
sales_customers.head()

,customer_id,first_name,surname,postcode,is_guest
0,7466.0,NaN,NaN,NaN,False
1,31266.0,NaN,NaN,NaN,False
2,534142828.0,NaN,NaN,NaN,False
3,1035.0,NaN,NaN,NaN,False
4,6985.0,NaN,NaN,NaN,False


In [52]:
# Now we add the in_purchase_data column we decided on for our schema:

sales_customers["in_purchase_data"] = True
sales_customers.head()

,customer_id,first_name,surname,postcode,is_guest,in_purchase_data
0,7466.0,NaN,NaN,NaN,False,True
1,31266.0,NaN,NaN,NaN,False,True
2,534142828.0,NaN,NaN,NaN,False,True
3,1035.0,NaN,NaN,NaN,False,True
4,6985.0,NaN,NaN,NaN,False,True


In [54]:
# ...ensure there is no trailing whitespace and that the names all use the same capitalization. 
# We can use pandas’ built-in .str accessor class, which lets us manipulate entire string columns.

for col in ["first_name", "surname"]:
    sales_customers[col] = sales_customers[col].str.lower().str.strip()

sales_customers["postcode"] = sales_customers["postcode"].str.strip()



In [56]:
print(sales_customers.shape)
sales_customers.head()

(33261, 6)


,customer_id,first_name,surname,postcode,is_guest,in_purchase_data
0,7466.0,NaN,NaN,NaN,False,True
1,31266.0,NaN,NaN,NaN,False,True
2,534142828.0,NaN,NaN,NaN,False,True
3,1035.0,NaN,NaN,NaN,False,True
4,6985.0,NaN,NaN,NaN,False,True


In [58]:
crm = pd.read_csv("C:/Users/gcart/OneDrive/Documents/dev/r/projs/book_reviews/solve_any_data_analysis_problem_projs/CH03_asboth_Solve_Any_Data_Analysis_Problem/data/crm_export.csv")
print(crm.shape)
crm.head()

(7825, 5)


,customer_id,first_name,surname,postcode,age
0,29223,Holly,Rogers,LS475RT,12
1,27826,Daniel,Owen,M902XX,5
2,7432,Eleanor,Russell,HR904ZA,34
3,2569,Paige,Roberts,DE732EP,61
4,9195,Matilda,Young,LS670FU,78


In [59]:
# chk for missing data
crm.isnull().sum()

customer_id    0
first_name     0
surname        0
postcode       0
age            0
dtype: int64

In [60]:
# ...ensuring that customer IDs are unique
crm.groupby("customer_id").size().loc[lambda x: x > 1]

Series([], dtype: int64)

In [61]:
# look at how many unique combinations of name, postcode, and age we have we can see this
print(len(crm))
print(len(crm.drop(columns="customer_id").drop_duplicates()))

7825
7419


In [62]:
# ...we also need to enhance the registered customers in the purchase history with details from 
# the CRM data. So far we only have IDs for those customers, but we need their names, postcodes, 
# and ages.

sales_and_crm_customers = sales_customers.merge(crm, on="customer_id", how="left")
print(len(sales_and_crm_customers))
sales_and_crm_customers.isnull().sum()

33261


customer_id          8300
first_name_x        24961
surname_x           24961
postcode_x          24961
is_guest                0
in_purchase_data        0
first_name_y        26147
surname_y           26147
postcode_y          26147
age                 26147
dtype: int64

In [63]:
# "What we have now is a dataset where 7,000 customer records are in columns ending in _y, which we should merge into the ones 
# marked _x, which contain customer data from guest checkouts. First, we define a filter to select only rows with a customer 
# ID, thus excluding guests, and rows with customer information in the _y suffixed columns."

merged_customers_filter = (
  (sales_and_crm_customers["customer_id"].notnull())
    & ((sales_and_crm_customers["first_name_y"].notnull())
      | (sales_and_crm_customers["surname_y"].notnull()))
)

In [64]:
# This filter can then be applied to first of all identify these rows as having been found in the CRM data.

sales_and_crm_customers.loc[merged_customers_filter, "in_crm_data"] = True
sales_and_crm_customers.loc[~merged_customers_filter, "in_crm_data"] = False
sales_and_crm_customers["in_crm_data"].value_counts()

False    26147
True      7114
Name: in_crm_data, dtype: int64

In [65]:
# This 7,114 figure tallies with what we observed earlier, that 7,000 rows have now had their customer information updated. 
# Time to copy over data from _y suffixed columns to our _x suffixed ones, and only keep the latter to get back to our 
# original schema. ...simply copied over the first name, surname, and postcode to overwrite the missing values in _x 
# suffixed columns with CRM customer data in the _y suffixed ones

sales_and_crm_customers.loc[merged_customers_filter, ["first_name_x","surname_x", "postcode_x"]] = (
    sales_and_crm_customers.loc[merged_customers_filter, ["first_name_y","surname_y", "postcode_y"]]
    .values
)

In [66]:
# remove the latter.

sales_and_crm_customers = (
    sales_and_crm_customers
    .drop(columns=["first_name_y", "surname_y", "postcode_y"])
    .rename(columns={
        "first_name_x": "first_name",
        "surname_x": "surname",
        "postcode_x": "postcode"
    })
)
sales_and_crm_customers.head()

,customer_id,first_name,surname,postcode,is_guest,in_purchase_data,age,in_crm_data
0,7466.0,NaN,NaN,NaN,False,True,NaN,False
1,31266.0,Harley,Palmer,HR250EJ,False,True,33.0,True
2,534142828.0,NaN,NaN,NaN,False,True,NaN,False
3,1035.0,NaN,NaN,NaN,False,True,NaN,False
4,6985.0,NaN,NaN,NaN,False,True,NaN,False


In [68]:
# check for, and add, customer details that exist in our CRM system but do not appear in our purchases. 

crm_ids_to_add = list(set(crm["customer_id"].unique()) - set(sales_and_crm_customers["customer_id"].unique()))
print(len(crm_ids_to_add))

711


In [69]:
#  concatenate/union the data with the customers corresponding to the IDs we have just selected.

purchase_and_crm_customers = (
    pd.concat([sales_and_crm_customers, crm[crm["customer_id"].isin(crm_ids_to_add)]],
              axis=0,
             ignore_index=True)
)

In [70]:
# One final aspect of this data to clean up is that for these new customers, we don’t have data for our source flags, 
# so we fill them in with default values. 
#    Customers added from the CRM data will have their in_crm_data flag set to True, 
#    in_purchase_data as False, 
#    and, since they weren’t guests, is_guest as False. 

sales_and_crm_customers["is_guest"] =          sales_and_crm_customers["is_guest"].fillna(False)
sales_and_crm_customers["in_purchase_data"]  = sales_and_crm_customers["in_purchase_data"].fillna(False)
sales_and_crm_customers["in_crm_data"] =       sales_and_crm_customers["in_crm_data"].fillna(True)
 
sales_and_crm_customers.isnull().sum()

customer_id          8300
first_name          17847
surname             17847
postcode            17847
is_guest                0
in_purchase_data        0
age                 26147
in_crm_data             0
dtype: int64

In [71]:
#  look at our customer database. 

customers = pd.read_csv("./data/customer_database.csv")
print(customers.shape)
customers.head()

(23476, 5)


,customer_id,first_name,surname,postcode,age
0,1641,Rhys,Richards,DE456EZ,45
1,24796,Maisie,Young,SW433XX,16
2,14358,Nathan,King,NW49TU,58
3,15306,Jack,Moore,NW908RR,26
4,24971,Alexander,Roberts,SW500HW,85


In [72]:
# Checking for missing data

customers.isnull().sum()

customer_id    0
first_name     0
surname        0
postcode       0
age            0
dtype: int64

In [75]:
# anitize our columns as we did for the CRM data

for col in ["first_name", "surname"]:
    customers[col] = customers[col].str.lower().str.strip()
 
    customers["postcode"] = customers["postcode"].str.strip()

In [76]:
# heck whether customer ID is unique, and how many unique combinations of customer information we have

customers.groupby("customer_id").size().loc[lambda x: x>1]

Series([], dtype: int64)

In [77]:
print("{} rows".format(len(customers)))
unique_customers = customers.drop(columns="customer_id").drop_duplicates()
print("{} unique combinations of customers".format(len(unique_customers)))

23476 rows
19889 unique combinations of customers


In [78]:
#  The next step is to merge the customer information into the growing customer data, by joining them together.

all_customers = sales_and_crm_customers.merge(customers, on="customer_id", how="left")
all_customers.head()

,customer_id,first_name_x,surname_x,postcode_x,is_guest,in_purchase_data,age_x,in_crm_data,first_name_y,surname_y,postcode_y,age_y
0,7466.0,NaN,NaN,NaN,False,True,NaN,False,eve,richards,HR90PT,45.0
1,31266.0,Harley,Palmer,HR250EJ,False,True,33.0,True,NaN,NaN,NaN,NaN
2,534142828.0,NaN,NaN,NaN,False,True,NaN,False,NaN,NaN,NaN,NaN
3,1035.0,NaN,NaN,NaN,False,True,NaN,False,luca,gibson,DE256NH,30.0
4,6985.0,NaN,NaN,NaN,False,True,NaN,False,mia,rogers,HR662RP,43.0


In [79]:
# identify which rows were successfully merged with the customer database, mark them with a final flag in_customer_data, 
# and copy those details over to the _x suffixed columns, before finally removing the redundant columns and arriving at 
# our final schema.

merged_customers_filter = (
    (all_customers["customer_id"].notnull())
    & ((all_customers["first_name_y"].notnull())
       | (all_customers["surname_y"].notnull()))
)
all_customers.loc[merged_customers_filter, "in_customer_data"] = True
all_customers.loc[~merged_customers_filter, "in_customer_data"] = False
all_customers["in_customer_data"].value_counts()

True     21342
False    11919
Name: in_customer_data, dtype: int64

In [80]:
# update the original customer details, those with a _x suffix, with details from the customer database

# a filter to mark customers whose details we copy over
update_filter = (
    (all_customers["in_customer_data"])
    & (all_customers["first_name_x"].isnull())
    & (all_customers["surname_x"].isnull())
)
 
# we then overwrite those customers’ details
all_customers.loc[update_filter, ["first_name_x", "surname_x", "postcode_x", "age_x"]] = (
    all_customers.loc[update_filter, ["first_name_y", "surname_y", "postcode_y", "age_y"]].values
)
 
# finally, merge the columns into the final schema
all_customers = (
    all_customers
    .drop(columns=["first_name_y", "surname_y", "age_y", "postcode_y"])
    .rename(columns={
        "first_name_x": "first_name",
        "surname_x": "surname",
        "age_x": "age",
        "postcode_x": "postcode"
    })
)

In [81]:
# add any customers present in the customer database who do not appear in the purchases table

customer_ids_to_add = list(set(customers["customer_id"].unique()) - set(all_customers["customer_id"].unique()))
print(len(customer_ids_to_add))

2134


In [82]:
all_customers = (
    pd.concat([all_customers, customers[customers["customer_id"].isin(customer_ids_to_add)]],
              axis=0,
              ignore_index=True)
)

In [83]:
# update the source flags to reflect their correct defaults if they are missing. Any newly added customers are not guests, 
# didn’t come from purchases or the CRM data, but are present in the customer database.

all_customers["is_guest"] = all_customers["is_guest"].fillna(False)
all_customers["in_purchase_data"]= all_customers["in_purchase_data"].fillna(False)
all_customers["in_crm_data"] = all_customers["in_crm_data"].fillna(False)
all_customers["in_customer_data"] = all_customers["in_customer_data"].fillna(True)

In [86]:
# inspect the final schema of our combined customer data model

print(all_customers.head())
print(all_customers.shape)

   customer_id first_name   surname postcode  is_guest  in_purchase_data  \
0       7466.0        eve  richards   HR90PT     False              True   
1      31266.0     Harley    Palmer  HR250EJ     False              True   
2  534142828.0        NaN       NaN      NaN     False              True   
3       1035.0       luca    gibson  DE256NH     False              True   
4       6985.0        mia    rogers  HR662RP     False              True   

    age  in_crm_data  in_customer_data  
0  45.0        False              True  
1  33.0         True             False  
2   NaN        False             False  
3  30.0        False              True  
4  43.0        False              True  
(35395, 9)


In [87]:
# customer types are mutually exclusive and their numbers should add up to the entire data model, and we can verify this

identified_customers = (
    all_customers[(all_customers["customer_id"].notnull())
                  & (all_customers["in_purchase_data"])
                  & ((all_customers["in_crm_data"])
                     | (all_customers["in_customer_data"]))]
)
 
guests = all_customers[all_customers["is_guest"]]
 
customer_ids_not_found = (
    all_customers[(all_customers["customer_id"].notnull())
                  & (all_customers["first_name"].isnull())
                  & (all_customers["surname"].isnull())]
)
 
customer_data_only = (
    all_customers[((all_customers["in_crm_data"])
                   | (all_customers["in_customer_data"])
                  )
                  & (all_customers["in_purchase_data"] == False)]
)
 
print(len(all_customers), len(identified_customers))
print(len(guests), len(customer_ids_not_found), len(customer_data_only))

35395 23713
8300 1248 2134


In [88]:
# The output is the following: 
#    35,395 records in the entire data model, 
#    23,713 of which are identified customers, 
#    8,300 guests, 
#    1,248 unidentified customer IDs, and 
#    2,134 customers with no purchases. 
# The first number is the sum of the others, so we can be confident we did not miss any of our eventualities, and they don’t 
#  overlap. At this stage, we know there might be some duplication, so we move on to the final part of modeling our customer 
#  data – entity resolution.

In [89]:
# we might have rows in our data model that are exact duplicates, which are easy to drop

print(len(all_customers))
all_customers = all_customers.drop_duplicates()
print(len(all_customers))

35395
35395


In [90]:
# look at the existing IDs to see the current range

all_customers["customer_id"].agg(["min", "max"])

min            1.0
max    566239774.0
Name: customer_id, dtype: float64

In [91]:
# use negative numbers instead to identify each unique guest, i.e. each combination of guest customer data points

all_guests = all_customers[all_customers["is_guest"]].copy()
new_ids = np.arange(-1, -(len(all_guests) + 1), -1)
all_customers.loc[all_customers["is_guest"], "customer_id"] = new_ids

In [ ]:
# Now our guest customers have IDs ranging from -1 to -8300

In [92]:
# find all customer IDs whose first name, last name, and postcode match another exactly

columns_to_consider = ["first_name", "surname", "postcode"]
 
duplicates = all_customers[all_customers.duplicated(subset=columns_to_consider, keep=False)]

In [95]:
# create a lookup dictionary where each customer ID is linked to all the other records which are its duplicates

duplicate_dict = duplicates.groupby(columns_to_consider)['customer_id'].apply(list).to_dict()

In [96]:
from itertools import islice
dict(islice(duplicate_dict.items(), 0, 2))

{('Aaron', 'James', 'LS992DB'): [10014.0, 30627.0],
 ('Abbie', 'Ward', 'SO310HE'): [21266.0, 29072.0]}

In [97]:
# store this list for accounts that have duplicates

all_customers['other_customer_ids'] = all_customers.apply(lambda x: duplicate_dict.get((x['first_name'],x['surname'], x['postcode'])), axis=1)

In [98]:
all_customers.head()

,customer_id,first_name,surname,postcode,is_guest,in_purchase_data,age,in_crm_data,in_customer_data,other_customer_ids
0,7466.0,eve,richards,HR90PT,False,True,45.0,False,True,None
1,31266.0,Harley,Palmer,HR250EJ,False,True,33.0,True,False,"[31266.0, 5411.0]"
2,534142828.0,NaN,NaN,NaN,False,True,NaN,False,False,None
3,1035.0,luca,gibson,DE256NH,False,True,30.0,False,True,None
4,6985.0,mia,rogers,HR662RP,False,True,43.0,False,True,None


In [99]:
# remove a customer’s own ID

def remove_own_record(row):
    ids = list(row["other_customer_ids"])
    ids.remove(row["customer_id"])
    return ids
 
all_customers.loc[all_customers["other_customer_ids"].notnull(),"duplicate_customer_ids"] = ( all_customers[all_customers["other_customer_ids"].notnull()].apply(remove_own_record, axis=1)
)



In [100]:
all_customers.head()

,customer_id,first_name,surname,postcode,is_guest,in_purchase_data,age,in_crm_data,in_customer_data,other_customer_ids,duplicate_customer_ids
0,7466.0,eve,richards,HR90PT,False,True,45.0,False,True,None,NaN
1,31266.0,Harley,Palmer,HR250EJ,False,True,33.0,True,False,"[31266.0, 5411.0]",[5411.0]
2,534142828.0,NaN,NaN,NaN,False,True,NaN,False,False,None,NaN
3,1035.0,luca,gibson,DE256NH,False,True,30.0,False,True,None,NaN
4,6985.0,mia,rogers,HR662RP,False,True,43.0,False,True,None,NaN


In [101]:
# create a column that gives each duplicate a rank, a row number in the order they are encountered.

all_customers["rank"] = all_customers.groupby(columns_to_consider).cumcount()+1

In [102]:
# to count distinct customer entities we can create our is_main flag to make the data model more obvious.

all_customers.loc[all_customers["rank"] == 1, "is_main"] = True
all_customers["is_main"] = all_customers["is_main"].fillna(False)
all_customers = all_customers.drop(columns="rank")

In [103]:
# Finally, we can count the records.

print(f"Total customers in DB: {len(all_customers)}")
print(f"Of which {len(all_customers[all_customers['is_main']])} are unique/main records")

Total customers in DB: 35395
Of which 29377 are unique/main records


In [104]:
# Let’s feed our data into this toolkit, a module called recordlinkage.

import recordlinkage

In [105]:
# “index” our dataset so that we don’t try to compare every pair of records to each other, but tell the code that two 
#  records at the same postcode should be tested for duplication

# Create an Index object
indexer = recordlinkage.Index()
# Mark postcode as a column to use for indexing
indexer.block('postcode')
# Apply the indexing to the data
candidate_links = indexer.index(all_customers.set_index("customer_id"))

In [106]:
#  Next, we set up the comparison rules – how should our records be compared against each other?

# Create a comparison object
compare = recordlinkage.Compare()
# Names should be fuzzy comparisons, anything over 85% similar is a match
compare.string('first_name', 'first_name', method='damerau_levenshtein',threshold=0.85, label="first_name")
compare.string('surname', 'surname', method='damerau_levenshtein',threshold=0.85, label="surname")
# Postcodes should match exactly
compare.exact('postcode', 'postcode', label="postcode")

<Compare>

In [107]:
# use our comparison rules to perform the pairwise comparisons.

compare_vectors = compare.compute(candidate_links, all_customers.set_index("customer_id"))

In [108]:
compare_vectors.head()

first_name  surname  postcode
customer_id_1 customer_id_2                               
7523.0        7466.0                0.0      0.0         1
7492.0        7466.0                0.0      0.0         1
              7523.0                0.0      0.0         1
7518.0        7466.0                0.0      0.0         1
              7523.0                0.0      0.0         1

In [109]:
# reduce this data down to only the cases where all comparisons returned a match

matches = compare_vectors[compare_vectors.sum(axis=1) == 3]

In [110]:
matches.head()

,,first_name,surname,postcode
customer_id_1,customer_id_2,,,
28583.0,7468.0,1.0,1.0,1
32670.0,7485.0,1.0,1.0,1
28195.0,7467.0,1.0,1.0,1
30670.0,7512.0,1.0,1.0,1
32205.0,7517.0,1.0,1.0,1


In [111]:
# merge these customer IDs back to the original dataset and see if we’ve improved our record linking attempts from before

# create a new DataFrame containing only the two columns of customer IDs
match_df = pd.DataFrame(
    data=matches.index.tolist(),
    columns=["customer_id_1", "customer_id_2"]
)
# join customer data on the first customer ID
matched = all_customers.merge(match_df, left_on="customer_id",right_on="customer_id_1", how="left")
# join customer data on the second customer ID
matched = matched.merge(match_df, left_on="customer_id",right_on="customer_id_2", how="left")

In [112]:
match_df.head()

,customer_id_1,customer_id_2
0,28583.0,7468.0
1,32670.0,7485.0
2,28195.0,7467.0
3,30670.0,7512.0
4,32205.0,7517.0
